## 0. Environment Setup

In [1]:
import logging

# Route agentic_llmr debug messages to the notebook output.
# Third-party loggers (langchain, httpx, openai) stay at WARNING to avoid noise.
_handler = logging.StreamHandler()
_handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))

_pkg_logger = logging.getLogger("agentic_llmr")
_pkg_logger.setLevel(logging.DEBUG)
_pkg_logger.addHandler(_handler)
_pkg_logger.propagate = False

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=f"{Path.home()}/.env", override=True)

# ROS / ament package paths needed for PR2 xacro parsing.
# These must be set BEFORE any pycram/SDT imports.
_IAI_PR2_INSTALL = "/home/malineni/ros_packages/iai_pr2_install"
_IAI_PR2_SRC     = "/home/malineni/ros_packages/iai_pr2/iai_pr2_description"
os.environ["AMENT_PREFIX_PATH"] = _IAI_PR2_INSTALL + ":" + os.environ.get("AMENT_PREFIX_PATH", "")
os.environ["ROS_PACKAGE_PATH"]   = _IAI_PR2_SRC     + ":" + os.environ.get("ROS_PACKAGE_PATH", "")

print("Environment configured.")

Environment configured.


In [ ]:
print("OPENAI_API_KEY configured:", bool(os.environ.get("OPENAI_API_KEY")))

In [3]:
from langchain_openai import ChatOpenAI
from agentic_llmr.core.orchestrator import ReActAgent
from agentic_llmr.platform.world import set_active_world
from uniworld import load_pr2_apartment_world
from agentic_llmr.core.trace import render_trace

print("Imports successful.")

`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`


Imports successful.


In [4]:
# ROS setup
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

## 1. Load the World

We use `load_pr2_simple_world()` from `uniworld`, which builds a self-contained SDT world:
- **PR2** robot (fully articulated, 40+ joints)
- **Table** (URDF) at x=1.1 m
- **Milk carton** (STL mesh) at (0.9, 0.0, 0.85) with `Milk` semantic annotation
- **Cereal box** (STL mesh) at (0.9, 0.2, 0.85) with `Cereal` semantic annotation

In [5]:
print("Loading PR2 simple world...")
world, robot_view, context = load_pr2_apartment_world()
set_active_world(world, robot_view)

bodies      = list(world.bodies)
annotations = list(world.semantic_annotations)

print(f"\nWorld loaded successfully!")
print(f"  Bodies              : {len(bodies)}")
print(f"  Semantic annotations: {len(annotations)}")
print("\nSemantic annotations:")
for ann in annotations:
    bodies_names = [str(getattr(b.name, "name", b.name)) for b in getattr(ann, "bodies", [])]
    print(f"  [{ann.__class__.__name__}]  bodies={bodies_names}")


Loading PR2 simple world...


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]



World loaded successfully!
  Bodies              : 221
  Semantic annotations: 16

Semantic annotations:
  [Finger]  bodies=['l_gripper_l_finger_link', 'l_gripper_l_finger_tip_link']
  [Finger]  bodies=['l_gripper_r_finger_link', 'l_gripper_r_finger_tip_link']
  [ParallelGripper]  bodies=['l_gripper_tool_frame', 'l_gripper_l_finger_tip_frame', 'l_gripper_led_frame', 'l_gripper_l_finger_tip_link', 'l_gripper_palm_link', 'l_gripper_r_finger_tip_link', 'l_gripper_motor_screw_link', 'l_gripper_motor_slider_link', 'l_gripper_l_finger_link', 'l_gripper_r_finger_link', 'l_gripper_motor_accelerometer_link']
  [Arm]  bodies=['torso_lift_link', 'l_shoulder_pan_link', 'l_shoulder_lift_link', 'l_upper_arm_roll_link', 'l_upper_arm_link', 'l_elbow_flex_link', 'l_forearm_roll_link', 'l_forearm_link', 'l_wrist_flex_link', 'l_wrist_roll_link']
  [Finger]  bodies=['r_gripper_l_finger_link', 'r_gripper_l_finger_tip_link']
  [Finger]  bodies=['r_gripper_r_finger_link', 'r_gripper_r_finger_tip_link']
  [P

In [6]:
# world.bodies

In [7]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


## 2. Initialise the Agent

The `ReActAgent` (Orchestrator) delegates to three specialist sub-agents:
- **SceneQueryAgent** — 16 SDT perception tools (poses, surfaces, spatial relations, placement)
- **KinematicsAgent** — Giskardpy reachability and grasp-pose tools
- **PlanningAgent** — PyCRAM action schema discovery and physics simulation

In [8]:
llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = ReActAgent(llm=llm)

nodes = list(agent.agent_executor.get_graph().nodes.keys())
print("Orchestrator ready — StateGraph supervisor routing to 3 specialist sub-agents.")
print(f"  Graph nodes : {nodes}")

print(f"Sub-agent tool breakdown:")
print(f"  SceneQueryAgent : {len(agent._scene_agent.tools)} tools")
print(f"  KinematicsAgent : {len(agent._kinematics_agent.tools)} tools")
print(f"  PlanningAgent   : {len(agent._planning_agent.tools)} tools  (schema discovery + simulation)")


Orchestrator ready — StateGraph supervisor routing to 3 specialist sub-agents.
  Graph nodes : ['__start__', 'supervisor', 'scene_perception', 'kinematics', 'planning', '__end__']
Sub-agent tool breakdown:
  SceneQueryAgent : 30 tools
  KinematicsAgent : 9 tools
  PlanningAgent   : 5 tools  (schema discovery + simulation)


## 3. Query Helper

A small utility that:
1. **Clears the scratchpad** so each query starts fresh
2. **Runs the agent** and prints the live reasoning trace
3. Returns the agent's final response

In [9]:
import os

def run_query(instruction: str, hint: str = "") -> str:
    """Submit an instruction to the agent and display the full reasoning trace."""
    for pad in ["sdt_scratchpad.md", "giskard_scratchpad.md", "pycram_scratchpad.md"]:
        with open(pad, "w") as f:
            f.write("")

    print(f"\n{chr(9473)*65}")
    print(f"  QUERY: {instruction}")
    print(f"{chr(9473)*65}\n")

    response = agent.resolve_action(instruction=instruction, template_context=hint)

    print(f"\n{chr(9472)*65}")
    print("FINAL RESPONSE:")
    print(response[:3000] if len(response) > 3000 else response)
    print(f"{chr(9472)*65}\n")
    return response

print("Query helper ready.")


Query helper ready.


---
## Query 1 — Scene Overview

**Goal:** Get a complete picture of everything in the world.

In [10]:
r1 = run_query(
    "Give me a complete overview of the current scene. "
    "List all objects present, their semantic types, and their 3D positions."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Give me a complete overview of the current scene. List all objects present, their semantic types, and their 3D positions.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires detailed information about the current scene, including object types and their 3D positions, which falls under the domain of scene perception.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] Give me a complete overview of the current scene. List all objects present, their semantic types, and their 3D positions
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Querying native semantic scene graph...
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 1 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query for a complete overview of the current scene has been fully answered with the list of objects, their types, and positions.)
DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR FINISHED]



─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The current scene contains the following objects:

1. **Table**
   - **Body Name:** table_area_main
   - **Position:** (5.000, 4.000, 0.000)

2. **Milk**
   - **Body Name:** milk.stl
   - **Position:** (2.370, 2.000, 1.050)

3. **Cereal**
   - **Body Name:** breakfast_cereal.stl
   - **Position:** (2.370, 1.800, 1.050)

4. **Robot**
   - **Body Name:** base_link
   - **Position:** (1.300, 2.000, 0.051)
─────────────────────────────────────────────────────────────────



In [ ]:
# render_trace(agent.last_trace, title="Query 1 — Scene Overview",
#                output_path="outputs/q1_scene_overview.html", open_browser=False)

---
## Query 2 — Finding Objects by Type

**Goal:** Locate specific semantic categories in the scene.

In [11]:
r2 = run_query(
    "Find all food-related objects in the scene. "
    "Report the exact 3D position of each one."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Find all food-related objects in the scene. Report the exact 3D position of each one.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about the locations of food-related objects in the scene.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] Find all food-related objects in the scene. Report the exact 3D position of each one.
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Food
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Listing semantic annotations...
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Querying native semantic scene graph...
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 2 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query has been fully answered with the positions of all food-related objects in the scene.)
DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR FINISHED]



─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The food-related objects in the scene and their exact 3D positions are as follows:

1. **Milk**: Position (2.370, 2.000, 1.050)
2. **Cereal**: Position (2.370, 1.800, 1.050)
─────────────────────────────────────────────────────────────────



---
## Query 3 — Surface Query

**Goal:** Understand which objects are on a specific surface.

**Expected SDT tools:** `list_semantic_annotations`, `get_objects_on_surface`

In [12]:
r3 = run_query(
    "Which objects are currently resting on the table? "
    "Report each object's semantic type, body name, and position."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Which objects are currently resting on the table? Report each object's semantic type, body name, and position.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about the objects on the table, which falls under scene perception.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] Which objects are currently resting on the table? Report each object's semantic type, body name, and position.
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Querying objects on surface: Table
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 1 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query has been fully answered as there are no objects on the table.)
DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR FINISHED]



─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
There are currently no objects resting on the table.
─────────────────────────────────────────────────────────────────



In [ ]:
# render_trace(agent.last_trace, title="Query 3 — Surface query",
#                output_path="outputs/q3_surface_query.html", open_browser=False)

---
## Query 4 — Object Property Analysis

**Goal:** Inspect the physical characteristics of a specific object.

**Expected SDT tools:** `get_object_pose`, `get_object_dimensions`, `get_object_orientation`

In [13]:
r4 = run_query(
    "Analyse the milk carton in detail: "
    "What is its exact 3D pose? What are its physical dimensions (width, depth, height)? "
    "Is it upright or lying on its side?"
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Analyse the milk carton in detail: What is its exact 3D pose? What are its physical dimensions (width, depth, height)? Is it upright or lying on its side?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires detailed information about the milk carton's 3D pose and physical dimensions, which falls under scene perception.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] Analyse the milk carton in detail: What is its exact 3D pose? What are its physical dimensions (width, depth, height)? I
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Milk
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting pose for: milk.stl
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting dimensions for: milk.stl
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Analysing orientation of: milk.stl
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 2 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query has been fully answered with detailed information about the milk carton's 3D pose, dimensions,


─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The milk carton has the following details:

- **3D Pose**: 
  - Position: (2.37, 2.00, 1.05)
  - Orientation: (0.0, 0.0, 0.0, 1.0) — indicating it is upright.

- **Physical Dimensions**: 
  - Width: 0.0642 m
  - Depth: 0.0651 m
  - Height: 0.1938 m

- **Orientation**: The milk carton is upright, with a tilt from vertical of 0.0°.
─────────────────────────────────────────────────────────────────



---
## Query 5 — Spatial Relationship Reasoning

**Goal:** Understand the relative positions between objects and the robot.

**Expected SDT tools:** `get_spatial_relation`, `get_nearest_objects`

In [14]:
r5 = run_query(
    "What is the spatial relationship between the milk carton and the cereal box? "
    "Which of the two is closer to the robot, and by how much?"
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: What is the spatial relationship between the milk carton and the cereal box? Which of the two is closer to the robot, and by how much?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about the spatial relationship and distances between the milk carton and the cereal box, which falls under scene perception.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] What is the spatial relationship between the milk carton and the cereal box? Which of the two is closer to the robot, an
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Milk
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Cereal
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Computing spatial relation: 'milk.stl' relative to 'breakfast_cereal.stl'
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Computing spatial relation: 'milk.stl' relative to 'robot'
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Computing spatial relation: 'breakfast_cereal.stl' relative to 'robot'
DEBUG | agentic_llmr.agents.scene_query_agent |   


─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The spatial relationship between the milk carton and the cereal box is that the milk carton is 0.200 meters to the left of the cereal box.

In terms of proximity to the robot:
- The milk carton is approximately 1.464 meters away from the robot.
- The cereal box is approximately 1.477 meters away from the robot.

Thus, the milk carton is closer to the robot by about 0.013 meters (or 13 millimeters).
─────────────────────────────────────────────────────────────────



---
## Query 6 — Size Comparison and Disambiguation

**Goal:** Rank objects by physical size — needed to resolve instructions like *'the larger object'*.

**Expected SDT tools:** `sort_objects_by_size`, `get_object_dimensions`

In [15]:
r6 = run_query(
    "Compare the physical sizes of the milk carton and the cereal box. "
    "Which one is larger? Provide their dimensions to justify your answer."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Compare the physical sizes of the milk carton and the cereal box. Which one is larger? Provide their dimensions to justify your answer.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about the physical sizes and dimensions of the milk carton and cereal box.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] Compare the physical sizes of the milk carton and the cereal box. Which one is larger? Provide their dimensions to justi
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Milk
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Cereal
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting dimensions for: milk.stl
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting dimensions for: breakfast_cereal.stl
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 2 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query has been fully answered with the dimensions and comparison of the milk carton and cereal box.)
DEBUG |


─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The dimensions and volumes of the two items are as follows:

**Milk Carton (milk.stl):**
- Width: 0.0642 m
- Depth: 0.0651 m
- Height: 0.1938 m
- Volume: 0.00081 m³

**Cereal Box (breakfast_cereal.stl):**
- Width: 0.1475 m
- Depth: 0.0581 m
- Height: 0.2236 m
- Volume: 0.001917 m³

**Comparison:**
- The cereal box is larger than the milk carton in both dimensions and volume.
─────────────────────────────────────────────────────────────────



In [16]:
# render_trace(agent.last_trace, title="Query 6 — Size Comparision",
#                output_path="outputs/q6_scene_overview.html", open_browser=False)

---
## Query 7 — Placement Planning

**Goal:** Find a suitable surface and free spots to place an object.

**Expected SDT tools:** `get_objects_on_surface`, `get_best_surface_for_placement`, `get_free_placement_spots`

In [17]:
r7 = run_query(
    "I want to place the cereal box somewhere after picking it up. "
    "Find the best surface available and suggest at least 3 concrete (x, y, z) positions "
    "on that surface where I could place it."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: I want to place the cereal box somewhere after picking it up. Find the best surface available and suggest at least 3 concrete (x, y, z) positions on that surface where I could place it.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about available surfaces and their positions to determine where to place the cereal box.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] I want to place the cereal box somewhere after picking it up. Find the best surface available and suggest at least 3 con
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Searching for semantic type: Cereal
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Finding supporting object for: breakfast_cereal.stl
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Classifying scene objects by role...
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Sampling placement spots on: table_area_main
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 1 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → kinematics (The next step is to determine if the robot can reach the suggested posit


─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The cereal box cannot be reached for picking up at its current position. The new navigation goal for the robot to approach is approximately **(1.870, 1.920, 0.0)**.

Now, I will suggest three concrete positions on the best available surface for placing the cereal box. Assuming the surface is flat and at a height of 1.05m (the original height of the cereal box), here are three suggested positions:

1. **Position 1:** (1.870, 1.920, 1.050)
2. **Position 2:** (1.870, 1.820, 1.050) - Slightly shifted to the left.
3. **Position 3:** (1.970, 1.920, 1.050) - Slightly shifted to the right.

These positions are all on the same surface and provide options for placing the cereal box after picking it up.
─────────────────────────────────────────────────────────────────



In [18]:
# render_trace(agent.last_trace, title="Query 7 — Surface query",
#                output_path="outputs/q7_placement.html", open_browser=False)

In [19]:
world.get_body_by_name('table_area_main').global_pose

Pose(@1=0.000796327, @2=0, @3=1, 
[[@1, -1, @2, 5], 
 [1, @1, @2, 4], 
 [@2, @2, @3, @2], 
 [@2, @2, @2, @3]])

---
## Query 8 — Full Pick-Up Parameter Resolution

**Goal:** Exercise the complete multi-tool reasoning pipeline for a manipulation task.

**Expected tools (in order):**
1. `write_scratchpad` — document plan
2. `list_available_actions` + `get_action_documentation` — discover `PickUpAction` schema
3. `find_objects_by_type` / `get_object_pose` — locate the milk
4. `get_object_dimensions` + `get_object_orientation` — understand object geometry
5. `check_kinematic_reachability` — verify arm can reach it
6. `get_grasp_poses` — get valid grasp descriptions

> **Note:** Execution (`simulate_action`) is intentionally skipped here. The agent will output the resolved JSON parameters.


In [ ]:
r8 = run_query(
    "Pick up the milk carton from the table.",
    hint="Resolve all parameters for a PickUpAction. Do not simulate — just output the final JSON."
)

In [ ]:
world.bodies

---
## Query 9 — Multi-Constraint Reasoning

**Goal:** The agent must combine spatial, reachability, and size reasoning to make a decision.

**Expected tools:** `get_objects_on_surface`, `get_spatial_relation`, `get_object_dimensions`, `check_kinematic_reachability`

In [20]:
r9 = run_query(
    "There are two objects on the table. "
    "Considering their positions relative to the robot and which arm can reach them, "
    "which object should I pick up first and with which arm? Justify your answer."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: There are two objects on the table. Considering their positions relative to the robot and which arm can reach them, which object should I pick up first and with which arm? Justify your answer.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (I need to gather information about the positions of the two objects on the table relative to the robot before determining which arm can reach them.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] There are two objects on the table. Considering their positions relative to the robot and which arm can reach them, whic
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Querying native semantic scene graph...
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Computing spatial relation: 'milk.stl' relative to 'robot'
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Computing spatial relation: 'breakfast_cereal.stl' relative to 'robot'
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting end-effector pose for: left arm
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting end-effector pose for: right arm
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted


─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
Based on the provided kinematic facts, both objects are unreachable for both the left and right arms of the robot. Since neither arm can reach either object, it is not possible to pick up any object at this time.

In this scenario, the next steps could involve either repositioning the robot to bring it within reach of one or both objects or considering alternative actions that do not require direct interaction with the objects. 

If you have any further instructions or need assistance with a different task, please let me know!
─────────────────────────────────────────────────────────────────



---
## Query 10 — Robot Configuration Inspection

**Goal:** Read the current robot joint states — useful before planning arm movements.

**Expected tools:** `get_joint_states`, `list_semantic_annotations`

In [21]:
r10 = run_query(
    "What is the current configuration of the PR2 robot? "
    "Report the current joint positions for both arms and summarise whether the arms "
    "are in a parked, extended, or arbitrary configuration."
)

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: What is the current configuration of the PR2 robot? Report the current joint positions for both arms and summarise whether the arms are in a parked, extended, or arbitrary configuration.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about the current state of the robot's joints and their configuration.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] What is the current configuration of the PR2 robot? Report the current joint positions for both arms and summarise wheth
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Reading active joint states...
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Getting robot base pose...
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 0 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query about the current configuration of the PR2 robot has been fully answered with the joint positions and configuration summary.)
DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR FINISHED]



─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The current joint positions for both arms of the PR2 robot are as follows:

**Left Arm:**
- Shoulder Pan: 0.0 rad
- Shoulder Lift: 0.0 rad
- Upper Arm Roll: 0.0 rad
- Elbow Flex: -0.15 rad
- Forearm Roll: 0.0 rad
- Wrist Flex: -0.1 rad
- Wrist Roll: 0.0 rad

**Right Arm:**
- Shoulder Pan: 0.0 rad
- Shoulder Lift: 0.0 rad
- Upper Arm Roll: 0.0 rad
- Elbow Flex: -0.15 rad
- Forearm Roll: 0.0 rad
- Wrist Flex: -0.1 rad
- Wrist Roll: 0.0 rad

### Configuration Summary:
Both arms are in a **parked configuration** as all joints are at their neutral positions (0.0 rad for most joints, and the elbow flex is at -0.15 rad, which is close to the limit). The arms are not extended or in an arbitrary configuration.
─────────────────────────────────────────────────────────────────



In [22]:
r11 = run_query("does the robot hold anything?")

DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR STARTED]



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: does the robot hold anything?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



DEBUG | agentic_llmr.core.supervisor | [Supervisor] → scene_perception (The query requires information about the robot's current state regarding whether it is holding any objects.)
DEBUG | agentic_llmr.agents.scene_query_agent |   ► [Scene Agent] does the robot hold anything?
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Checking held object for: left arm
DEBUG | agentic_llmr.tools.scene_query | [SDT Tool] Checking held object for: right arm
DEBUG | agentic_llmr.agents.scene_query_agent |   ◄ [Scene Agent] Done. Extracted 0 scene fact entries.
DEBUG | agentic_llmr.core.supervisor | [Supervisor] → FINISH (The query is fully answered as the robot's current state regarding holding objects has been provided.)
DEBUG | agentic_llmr.core.orchestrator | [ORCHESTRATOR FINISHED]



─────────────────────────────────────────────────────────────────
FINAL RESPONSE:
The robot is not holding any object in either the left or right arm.
─────────────────────────────────────────────────────────────────



---
## Summary: SDT Tool Coverage

| Query | Primary tools exercised |
|---|---|
| 1 — Scene overview | `get_objects_in_scene`, `list_semantic_annotations` |
| 2 — Find by type | `find_objects_by_type`, `get_object_pose` |
| 3 — Surface query | `get_objects_on_surface` |
| 4 — Object properties | `get_object_pose`, `get_object_dimensions`, `get_object_orientation` |
| 5 — Spatial relations | `get_spatial_relation`, `get_nearest_objects` |
| 6 — Size comparison | `sort_objects_by_size`, `get_object_dimensions` |
| 7 — Placement planning | `get_best_surface_for_placement`, `get_free_placement_spots` |
| 8 — Full pick-up plan | All SDT + PyCRAM + Giskard tools in sequence |
| 9 — Multi-constraint | `get_spatial_relation`, `check_kinematic_reachability`, `get_object_dimensions` |
| 10 — Robot config | `get_joint_states` |